# Problem 4: Resource Monitoring & Job Diagnostics

---

> Place all files for this problem in a directory called `problem_4`.

## Overview

One of the most important skills in HPC is diagnosing **why a job failed** and learning to request the right resources. In this problem, you will intentionally submit jobs with insufficient resources, observe how they fail, learn to read SLURM error output, and use diagnostic tools to understand job performance. These are skills you will rely on heavily when running larger jobs later in this course.


## Part A: Create a Resource-Hungry Script

Write a Python script called `memory_hog.py` that:

1. Accepts a command-line argument for the target size in MB (e.g., `python memory_hog.py 512`)
2. Allocates a large list in memory to consume approximately that many megabytes (Hint: a list of `n` integers where `n = target_mb * 1024 * 1024 // 8` will consume roughly `target_mb` of memory)
3. Computes the sum of all numbers in the list and prints the result
4. Prints the amount of memory consumed (use `import sys; sys.getsizeof()` or the `resource` module)
5. Sleeps for 30 seconds (to give you time to observe the job while running)
6. Prints "Completed successfully"


## Part B: Trigger a Memory Failure

1. Write a batch script called `too_little_mem.sbatch` that:
   - Requests only **100 MB** of memory (`--mem=100M`)
   - Runs `memory_hog.py` with a target of **512 MB**
   - Sets a time limit of 5 minutes

2. Submit the job and wait for it to complete (it should fail).

3. Examine the `.err` file and run:

```bash
sacct --jobs=<job_id> --format=JobID,JobName,State,ExitCode,MaxRSS,ReqMem,Elapsed
```

4. In a file called `answers.md`, answer:
   - What was the **State** of the job? What does this state mean?
   - What **signal** killed the job? (Hint: look for `OOM` — Out of Memory)
   - What does `sacct` report for **MaxRSS** versus **ReqMem**?


## Part C: Trigger a Time Limit Failure

1. Write a batch script called `too_little_time.sbatch` that:
   - Requests adequate memory (**1G**)
   - Sets the time limit to only **10 seconds** (`--time=00:00:10`)
   - Runs `memory_hog.py` with a target of **256 MB** (which sleeps for 30 seconds)

2. Submit the job and wait for it to complete (it should be killed for exceeding the time limit).

3. Examine the output and run `sacct` again.

4. In `answers.md`, answer:
   - What **State** does SLURM report for a job that exceeded its time limit?
   - What is the practical consequence of always requesting the maximum time limit on a shared cluster? Why not just ask for 24 hours every time?


## Part D: Right-Sizing Your Resources

Now fix both failures:

1. Write a batch script called `right_sized.sbatch` that requests appropriate memory and time for running `memory_hog.py` with a target of **512 MB**.

2. Submit the job and verify it completes successfully.

3. After the job finishes, use `sacct` to check the actual resource usage:

```bash
sacct --jobs=<job_id> --format=JobID,JobName,State,ExitCode,MaxRSS,ReqMem,Elapsed,TotalCPU
```

4. In `answers.md`, answer:
   - What was the actual peak memory (**MaxRSS**) and how does it compare to what you requested?
   - What is a reasonable rule of thumb for how much extra memory to request beyond what you expect to use?
   - Why is it important to right-size resource requests on a shared cluster rather than massively over-requesting?


## Part E: Job Monitoring

While a job is running, you can observe it in real time. Submit `right_sized.sbatch` again and while it is running:

1. Use `squeue` to check its status.
2. SSH into the compute node where the job is running and use `top` or `htop` to observe the process:

```bash
ssh <node_name>
top -u <your_cnetid>
```

3. In `answers.md`, describe what you see in `top`. Can you identify your Python process? How much memory and CPU is it using?


## Deliverables

Your `problem_4` directory should contain:

```
problem_4/
├── answers.md
├── memory_hog.py
├── too_little_mem.sbatch
├── too_little_time.sbatch
└── right_sized.sbatch
```

**Submitting.** Put everything for this problem in `problem_4/` in your repository, with your results easy to find: your scripts, the output files your answers refer to, and `answers.md`. You can work in your clone on Midway3 and commit and push from there, or copy your work back to your laptop (for example with `scp`) and push from there. Either is fine.
